### Neues FactSheet `Application` in LeanIX erstellen/hinzufügen 
#### Nutzen: Teilautomatisierung des Prozesses `Neue Applikation im Applikationsinventar erfassen`
#### Pflichtfelder für FactSheet `Application` (Stand: 01.07.2025):
- Name
- Beschreibung
- Applikaitonstyp
- Fachliche Eignung
- Geschäftskritikalität 
- Datenschutz:
    - Speicherung DSG relevante Daten
- Lebenszyklus
- Pflichtabonnements:
    - Zuständig: AV Fachlich
- Pflichtrelationen:
    - Benutzergruppen
    - IT-Komponenten
 
- weitere Felder können hinzügefügt werden

### Imports

In [ ]:
import os 
import sys

# Build an absolute path from this notebook's parent directory
module_path = os.path.abspath(os.path.join('..'))

# Add `module_path` to sys.path if not already present
if module_path not in sys.path:
    sys.path.append(module_path)

#  import the desired modules
import new_factsheet as nf
import graphql_leanix_utils as gqlix

# **Note**: 
- API tokens are required to access LeanIX services. These should be placed in the `.env` file in the root directory.
- **It is recommended to experiment using the `LEANIX_API_TOKEN_SANDBOX` before applying any changes using `LEANIX_API_TOKEN`**.
    In`.env` file:
    - `LEANIX_API_TOKEN_SANDBOX`: API Token for Sandbox 
    - `LEANIX_API_TOKEN`: API Token


### FactSheet `Application` direkt aus eine `queryX.txt`-Datei erstellen 
- Beispiel: Aus der Datei `create_factsheet_from_query.txt` im `queries_txt` Ordner

In [ ]:
factsheet_from_query = nf.create_new_app_from_query_file(query_file= "create_factsheet_from_query")
factsheet_from_query

### Abonnoments (Applikationsverantwortliche Person) zu FactSheet `Application` hinzufügen
- Beispiel: `Max Musterman `als `AV Fachlich` und `AV Technisch` für die Applikation `_TEST CREATE NEW Application`

In [ ]:
# get leanIX Id of the newly created app 
app_id = gqlix.get_app_id_by_name(name= "_TEST CREATE NEW Application")

# there is an id for each role, relevent roles are: 
#"AV Fachlich" = "7d3da7df-89d0-4e72-a61b-5693cfdf33ff"
#"AV Technisch" = "6f8aa477-1b9b-44b7-af6c-237c25e5adf0"
#"GPV" = "99c9c884-b163-4d0c-993f-10c95b04e859"
subscription = nf.add_subscriptions(factSheet_id= app_id, user_email= "max.musterman@email.com", role_type="RESPONSIBLE", 
                                    user_roles= ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"])
subscription

---

### FactSheet `Application` aus einem modularen Query-Template erstellen 
- Funktion: `create_new_app()`
- Beispiel: Query Template `create_app_factsheet_template.txt` im `queries_txt` Ordner
    Felder:
    - name: str,
    - type: str,
    - apptype: str,
    - description: str,
    - businessCriticality: str,
    - functionalSuitability: str,
    - datamaster: bool,
    - subscriptions: SubscriptionDict,
    - user_groups_ids: list,
    - ITComponents_ids: list,
    - lifecycle_phase: LifecyclePhaseDict


In [ ]:
# businessCriticality can have five different keys: ["__missing__", "administrativeService", "businessCritical", "businessOperational", "missionCritical"]
# Specify the businessCriticality
_businessCriticality = "administrativeService"

# functional_suitability can have five different keys: ["__missing__", "appropriate", "insufficient", "perfect", "unreasonable"]
# Specify the functionalSuitabilityt
_functionalSuitability = "appropriate"

# UserGroups can only be added using their LeanIX Id
# specify a list of UserGroups to be linked 
_user_groups_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# ITComponents can only be added using their LeanIX Id
# specify list of ITComponents to be linked 
_ITComponents_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# Specify the lifecycle phase using a custom TypedDict
_lifecycle_phase : nf.LifecyclePhaseDict = {"type": "active", "startDate": "2021-01-01"}

# Specify subscriptions using a custom TypedDict
# Note: "factSheet_id" is not nedeed in this case (in method "create_new_app()") because we are creating a new FactSheet.
#       Check method "create_new_app()" for implementation details
_subscriptions : nf.SubscriptionDict = {"factSheet_id": "", "user_email": "max.musterman@email.com", 
                                     "role_type": "RESPONSIBLE", 
                                     "user_roles":["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]}


new_app_factSheet_1 = nf.create_new_app(name= "_TEST CREATE NEW Application From Template", 
                           type= "Application", apptype="web_app", description= "Description ...", 
                           businessCriticality= _businessCriticality, functionalSuitability= _functionalSuitability,
                           datamaster= False, subscriptions= _subscriptions,
                           user_groups_ids= _user_groups_ids,
                           ITComponents_ids= _ITComponents_ids, 
                           lifecycle_phase=_lifecycle_phase)

new_app_factSheet_1

### FactSheet `Application` aus einem modularen Query-Template erstellen (Weiterere Felder hinzufügen)
- Funktion: `create_new_app()`
- Beispiel: Query Template `create_app_factsheet_template.txt` im `queries_txt` Ordner
    Pfilchtfelder:
    - name: str,
    - type: str,
    - apptype: str,
    - description: str,
    - businessCriticality: str,
    - functionalSuitability: str,
    - datamaster: bool,
    - subscriptions: SubscriptionDict,
    - user_groups_ids: list,
    - ITComponents_ids: list,
    - lifecycle_phase: LifecyclePhaseDict
    - additional_app_attributes: list[FactSheetAttributeDict] = None

    weiterere Felder: 
    - completeClientDatabase (Kundendatenumfang)
    - massDataQueries (Massendatenabfragen)

In [ ]:
# businessCriticality can have five different keys: ["__missing__", "administrativeService", "businessCritical", "businessOperational", "missionCritical"]
# Specify the businessCriticality
_businessCriticality = "administrativeService"

# functional_suitability can have five different keys: ["__missing__", "appropriate", "insufficient", "perfect", "unreasonable"]
# Specify the functionalSuitabilityt
_functionalSuitability = "appropriate"

# UserGroups can only be added using their LeanIX Id
# specify a list of UserGroups to be linked 
_user_groups_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# ITComponents can only be added using their LeanIX Id
# specify list of ITComponents to be linked 
_ITComponents_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# Specify the lifecycle phase using a custom TypedDict
_lifecycle_phase : nf.LifecyclePhaseDict = {"type": "active", "startDate": "2021-01-01"}

# Specify subscriptions using a custom TypedDict
# Note: "factSheet_id" is not nedeed in this case (in method "create_new_app()") because we are creating a new FactSheet.
#       Check method "create_new_app()" for implementation details
_subscriptions : nf.SubscriptionDict = {"factSheet_id": "", "user_email": "max.mustermann@bank.com", 
                                     "role_type": "RESPONSIBLE", 
                                     "user_roles":["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]}

# Add additional attributes 
# Specify additional attributes using a custom TypedDict
# Example: Add attribute "completeClientDatabase = no" (Kundendatenumfang) and "massDataQueries = no" (Massendatenabfragen), ...

# datamaster (Datenschutzgesetz) can have five different keys: ["False", "True", "__missing__"]
# completeClientDatabase (Kundendatenumfang) can have five different keys: ["False", "True", "__missing__"]
_completeClientDatabase: nf.FactSheetAttributeDict={"attribute_name": "completeClientDatabase", "attribute_value": "no"}
# massDataQueries (Massendatenabfragen) can have different keys: ["yes", "no", "__missing__", "export"]
_massDataQueries: nf.FactSheetAttributeDict={"attribute_name": "massDataQueries", "attribute_value": "no"}
# deletionProcess (Löschmechanismus) can have five different keys: ["__missing__", "none", "singleDeletion", "massDeletion"]
_deletionProcess: nf.FactSheetAttributeDict={"attribute_name": "deletionProcess", "attribute_value": "massDeletion"}
# holdOffTime (Vorhaltezeit) can have five different keys: ["__missing__", "notApplicable", "equalOrLessThan3Months", "equalOrLessThan12Months", "greaterThan12Months"]
_holdOffTime : nf.FactSheetAttributeDict={"attribute_name": "holdOffTime", "attribute_value": "equalOrLessThan3Months"}
# personDataProtectionAgreements (Vereinbarung zur Datensicherheit und Datenschutz mit Externen) can have five different keys: ["notRequired", "__missing__", "inPlace", "notInPlace", "onParent"]
_personDataProtectionAgreements : nf.FactSheetAttributeDict={"attribute_name": "personDataProtectionAgreements", "attribute_value": "notRequired"}
_additonal_app_attributes = [_completeClientDatabase, _massDataQueries, _deletionProcess, _holdOffTime, _personDataProtectionAgreements]

new_app_factSheet_2 = nf.create_new_app(name= "_TEST CREATE NEW Application From Template With Additional Attributes", 
                           type= "Application", apptype="web_app", description= "Description ...", 
                           businessCriticality= _businessCriticality, functionalSuitability= _functionalSuitability,
                           datamaster= False, subscriptions= _subscriptions,
                           user_groups_ids= _user_groups_ids,
                           ITComponents_ids= _ITComponents_ids, 
                           lifecycle_phase=_lifecycle_phase,
                           additional_app_attributes=_additonal_app_attributes)

new_app_factSheet_2

In [ ]:
# businessCriticality can have five different keys: ["__missing__", "administrativeService", "businessCritical", "businessOperational", "missionCritical"]
# Specify the businessCriticality
_businessCriticality = "administrativeService"

# functional_suitability can have five different keys: ["__missing__", "appropriate", "insufficient", "perfect", "unreasonable"]
# Specify the functionalSuitabilityt
_functionalSuitability = "appropriate"

# UserGroups can only be added using their LeanIX Id
# specify a list of UserGroups to be linked 
_user_groups_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# ITComponents can only be added using their LeanIX Id
# specify list of ITComponents to be linked 
_ITComponents_ids = ["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]

# Specify the lifecycle phase using a custom TypedDict
_lifecycle_phase : nf.LifecyclePhaseDict = {"type": "active", "startDate": "2021-01-01"}

# Specify subscriptions using a custom TypedDict
# Note: "factSheet_id" is not nedeed in this case (in method "create_new_app()") because we are creating a new FactSheet.
#       Check method "create_new_app()" for implementation details
_subscriptions : nf.SubscriptionDict = {"factSheet_id": "", "user_email": "max.musterman@email.com", 
                                     "role_type": "RESPONSIBLE", 
                                     "user_roles":["XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX", "XXXXXXXX-XXXX-XXXX-XXXX-XXXXXXXXXXXX"]}

# Add additional attributes 
# Specify additional attributes using a custom TypedDict
# Example: Add attribute "completeClientDatabase = no" (Kundendatenumfang) and "massDataQueries = no" (Massendatenabfragen), ...

# datamaster (Datenschutzgesetz) can have five different keys: ["False", "True", "__missing__"]
# completeClientDatabase (Kundendatenumfang) can have five different keys: ["False", "True", "__missing__"]
_completeClientDatabase: nf.FactSheetAttributeDict={"attribute_name": "completeClientDatabase", "attribute_value": "no"}
# massDataQueries (Massendatenabfragen) can have different keys: ["yes", "no", "__missing__", "export"]
_massDataQueries: nf.FactSheetAttributeDict={"attribute_name": "massDataQueries", "attribute_value": "no"}
# deletionProcess (Löschmechanismus) can have five different keys: ["__missing__", "none", "singleDeletion", "massDeletion"]
_deletionProcess: nf.FactSheetAttributeDict={"attribute_name": "deletionProcess", "attribute_value": "massDeletion"}
# holdOffTime (Vorhaltezeit) can have five different keys: ["__missing__", "notApplicable", "equalOrLessThan3Months", "equalOrLessThan12Months", "greaterThan12Months"]
_holdOffTime : nf.FactSheetAttributeDict={"attribute_name": "holdOffTime", "attribute_value": "equalOrLessThan3Months"}
# personDataProtectionAgreements (Vereinbarung zur Datensicherheit und Datenschutz mit Externen) can have five different keys: ["notRequired", "__missing__", "inPlace", "notInPlace", "onParent"]
_personDataProtectionAgreements : nf.FactSheetAttributeDict={"attribute_name": "personDataProtectionAgreements", "attribute_value": "notRequired"}
_additonal_app_attributes = [_completeClientDatabase, _massDataQueries, _deletionProcess, _holdOffTime, _personDataProtectionAgreements]

new_app_factSheet_2 = nf.create_new_app(name= "_TEST CREATE NEW Application From Template With Additional Attributes", 
                           type= "Application", apptype="web_app", description= "Description ...", 
                           businessCriticality= _businessCriticality, functionalSuitability= _functionalSuitability,
                           datamaster= False, subscriptions= _subscriptions,
                           user_groups_ids= _user_groups_ids,
                           ITComponents_ids= _ITComponents_ids, 
                           lifecycle_phase=_lifecycle_phase,
                           additional_app_attributes=_additonal_app_attributes)

new_app_factSheet_2